# Identificación de Sesgos y Preprocesamiento

**Dataset:** `merged_hotels.json` — 680 hoteles | Barcelona · Madrid · Bilbao

Este notebook cubre dos bloques:
- **Bloque I — Identificación de Sesgos:** documenta los sesgos estructurales del corpus antes de cualquier transformación
- **Bloque J — Preprocesamiento:** implementa el pipeline de limpieza paso a paso, con métricas de impacto en cada etapa

In [1]:
import json, re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

BASE_PATH  = "/Users/alessiadelconte/codici/progetto NPL"
INPUT_FILE = os.path.join(BASE_PATH, "DATASET/merged_hotels.json")
OUTPUT_DIR = os.path.join(BASE_PATH, "output_sesgos_preprocesamiento")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CITY_COLORS = {'Barcelona': '#2196F3', 'Madrid': '#FF9800', 'Bilbao': '#4CAF50'}
SRC_COLORS  = {'booking': '#9C27B0', 'hotels_com': '#F44336', 'expedia': '#00BCD4'}
SENT_COLORS = {'Negativo': '#F44336', 'Neutro': '#FFC107', 'Positivo': '#4CAF50'}

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

# DataFrame base
rows = []
for h in data:
    try:
        price = float(str(h.get('price', 0)).replace(',', ''))
        price = price if price > 0 else np.nan
    except:
        price = np.nan
    for r in h.get('reviews', []):
        if isinstance(r, dict):
            rows.append({
                'hotel':      h.get('name', ''),
                'city':       h.get('city', ''),
                'source':     r.get('source', ''),
                'comment':    str(r.get('comment', '')),
                'score':      pd.to_numeric(r.get('score'), errors='coerce'),
                'date':       r.get('date', ''),
                'price':      price,
                'n_services': len(h.get('services', [])),
            })

df_raw = pd.DataFrame(rows)
df_raw['n_words'] = df_raw['comment'].str.split().str.len()
print(f'✅ Dataset cargado: {len(df_raw)} reseñas brutas de {len(data)} hoteles')

✅ Dataset cargado: 3881 reseñas brutas de 680 hoteles


---
## Bloque I — Identificación de Sesgos

Un sesgo es cualquier característica sistemática del corpus que distorsione la capacidad del modelo para generalizar.
Se identifican siete sesgos estructurales en este dataset.

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# FIG I1 — SESGO DE PLATAFORMA: scores disponibles por fuente
# ══════════════════════════════════════════════════════════════════════════
df_s = df_raw.dropna(subset=['score'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('I1 — Sesgo de Plataforma: Scores Disponibles por Fuente',
             fontsize=16, fontweight='bold')

for ax, src in zip(axes, ['booking', 'expedia', 'hotels_com']):
    sub = df_s[df_s['source'] == src]
    score_cnt = sub['score'].value_counts().sort_index()
    all_scores = [2.0, 4.0, 6.0, 7.0, 8.0, 9.0, 10.0]
    counts = [score_cnt.get(s, 0) for s in all_scores]
    bar_colors = ['#F44336' if s <= 6 else '#FFC107' if s <= 8 else '#4CAF50'
                  for s in all_scores]
    bars = ax.bar([str(int(s)) for s in all_scores], counts,
                  color=bar_colors, edgecolor='white')
    ax.set_title(f'{src.replace("_",".").title()}\n(n={len(sub)})', fontweight='bold',
                 color=SRC_COLORS.get(src, '#333'))
    ax.set_xlabel('Score')
    ax.set_ylabel('N.º reseñas')
    for bar, val in zip(bars, counts):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, val + 5,
                    str(val), ha='center', fontsize=8)

# Nota explicativa
fig.text(0.5, -0.02,
         '⚠️  Booking no incluye scores 2 ni 4 → sesgo de truncamiento en la cola negativa',
         ha='center', fontsize=10, color='#B71C1C', style='italic')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'I1_sesgo_plataforma_scores.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Fig I1 guardada ✓')
print('\n📊 Tabla: scores disponibles por plataforma')
print(pd.crosstab(df_s['source'], df_s['score']).to_string())

Fig I1 guardada ✓

📊 Tabla: scores disponibles por plataforma
score       2.0   4.0   6.0   7.0   8.0   9.0   10.0
source                                              
booking        0     0   295   302   308   300   280
expedia       19    33    66     0   206     0   660
hotels_com    19    34    71     0   234     0  1050


In [3]:
# ══════════════════════════════════════════════════════════════════════════
# FIG I2 — SESGO DE POSITIVIDAD (positivity bias)
# ══════════════════════════════════════════════════════════════════════════
# Quienes publican una reseña no son una muestra aleatoria de todos los huéspedes.
# Los huéspedes muy satisfechos o muy insatisfechos tienen más probabilidad de escribir.
# En este corpus, el 59% de las reseñas tienen score 9 o 10.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('I2 — Sesgo de Positividad (Positivity Bias)', fontsize=16, fontweight='bold')

# I2a: Distribución global de scores
score_cnt = df_s['score'].value_counts().sort_index()
all_scores = [2.0, 4.0, 6.0, 7.0, 8.0, 9.0, 10.0]
counts = [score_cnt.get(s, 0) for s in all_scores]
bar_colors = ['#F44336' if s <= 6 else '#FFC107' if s <= 8 else '#4CAF50' for s in all_scores]
bars = axes[0].bar([str(int(s)) for s in all_scores], counts, color=bar_colors, edgecolor='white')
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 15,
                 f'{val}\n({val/len(df_s)*100:.0f}%)', ha='center', fontsize=8)
axes[0].set_title('Distribución Global: 59% de reseñas con score 9-10', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('N.º reseñas')
patches = [
    mpatches.Patch(color='#F44336', label='Negativo (≤6)'),
    mpatches.Patch(color='#FFC107', label='Neutro (7-8)'),
    mpatches.Patch(color='#4CAF50', label='Positivo (9-10)'),
]
axes[0].legend(handles=patches, fontsize=8)

# I2b: Score medio por longitud del comentario
df_s['len_bucket'] = pd.cut(df_s['n_words'],
    bins=[0, 5, 15, 30, 200],
    labels=['< 5 palabras', '5-15 palabras', '15-30 palabras', '> 30 palabras'])
score_by_len = df_s.groupby('len_bucket', observed=True)['score'].mean()
bars2 = axes[1].bar(score_by_len.index, score_by_len.values,
                    color=['#FFCDD2', '#FFCC80', '#A5D6A7', '#64B5F6'], edgecolor='white')
axes[1].set_ylim(7.5, 9.5)
for bar, val in zip(bars2, score_by_len.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.04,
                 f'{val:.2f}', ha='center', fontweight='bold')
axes[1].set_title('Score Medio por Longitud del Comentario\n(los textos largos tienen score más bajo → insatisfacción motiva más escritura)',
                  fontweight='bold')
axes[1].set_ylabel('Score medio')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'I2_sesgo_positividad.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Fig I2 guardada ✓')

Fig I2 guardada ✓


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# FIG I3 — SESGO GEOGRÁFICO Y DE COBERTURA
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('I3 — Sesgo Geográfico y de Cobertura', fontsize=16, fontweight='bold')

# I3a: Hoteles con y sin reseñas por ciudad
hotel_stats = {}
for h in data:
    city = h.get('city', '')
    has_rev = len(h.get('reviews', [])) > 0
    if city not in hotel_stats:
        hotel_stats[city] = {'con': 0, 'sin': 0}
    if has_rev:
        hotel_stats[city]['con'] += 1
    else:
        hotel_stats[city]['sin'] += 1

cities_order = ['Barcelona', 'Madrid', 'Bilbao']
con_vals = [hotel_stats[c]['con'] for c in cities_order]
sin_vals = [hotel_stats[c]['sin'] for c in cities_order]
x = np.arange(3)
axes[0].bar(x - 0.2, con_vals, 0.4, label='Con reseñas', color='#4CAF50', edgecolor='white')
axes[0].bar(x + 0.2, sin_vals, 0.4, label='Sin reseñas', color='#F44336', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cities_order)
axes[0].set_title('Hoteles con y sin Reseñas por Ciudad\n(63 hoteles de Madrid sin reseñas: sesgo de cobertura)',
                  fontweight='bold')
axes[0].set_ylabel('N.º hoteles')
axes[0].legend()
for i, (c, s) in enumerate(zip(con_vals, sin_vals)):
    axes[0].text(i - 0.2, c + 1, str(c), ha='center', fontsize=8, fontweight='bold')
    axes[0].text(i + 0.2, s + 1, str(s), ha='center', fontsize=8, fontweight='bold')

# I3b: Distribución reseñas por hotel
rev_per_hotel = [len(h.get('reviews', [])) for h in data if h.get('reviews')]
axes[1].hist(rev_per_hotel, bins=15, color='#5C6BC0', edgecolor='white')
axes[1].axvline(np.mean(rev_per_hotel), color='red', linestyle='--',
                label=f'Media: {np.mean(rev_per_hotel):.1f}')
axes[1].set_title('Distribución Reseñas por Hotel\n(mayoría entre 5 y 8 → sesgo de muestreo uniforme)',
                  fontweight='bold')
axes[1].set_xlabel('N.º reseñas')
axes[1].legend()

# I3c: Sesgo temporal — solo Hotels.com tiene fechas
date_cov = {'booking': 0, 'expedia': 0, 'hotels_com': 99}
bars = axes[2].bar(list(date_cov.keys()), list(date_cov.values()),
                   color=[SRC_COLORS.get(s, '#607D8B') for s in date_cov], edgecolor='white')
axes[2].set_ylim(0, 115)
for bar, val in zip(bars, date_cov.values()):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'{val}%', ha='center', fontweight='bold')
axes[2].set_title('Cobertura de Fecha por Plataforma\n(sesgo temporal: solo Hotels.com permite análisis cronológico)',
                  fontweight='bold')
axes[2].set_ylabel('% reseñas con fecha')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'I3_sesgo_geografico_cobertura.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Fig I3 guardada ✓')

Fig I3 guardada ✓


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# FIG I4 — SESGO LINGÜÍSTICO Y DE LONGITUD
# ══════════════════════════════════════════════════════════════════════════
def detect_lang(text):
    t = str(text).lower()
    es = sum(1 for w in ['muy','habitación','ubicación','limpieza','personal','desayuno',
                         'cómodo','también','calidad','precio'] if w in t)
    en = sum(1 for w in ['room','staff','great','good','clean','location','nice',
                         'stay','comfortable','breakfast'] if w in t)
    if es == 0 and en == 0: return 'other'
    return 'es' if es >= en else 'en'

df_s['lang'] = df_s['comment'].apply(detect_lang)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('I4 — Sesgo Lingüístico y de Longitud', fontsize=16, fontweight='bold')

# I4a: Distribución idioma
LANG_COLORS = {'es': '#2196F3', 'en': '#FF9800', 'other': '#9E9E9E'}
lang_cnt = df_s['lang'].value_counts()
axes[0].pie(
    [lang_cnt.get('es', 0), lang_cnt.get('en', 0), lang_cnt.get('other', 0)],
    labels=['Español\n(67.7%)', 'Inglés\n(4.2%)', 'Otros\n(28.1%)'],
    colors=[LANG_COLORS['es'], LANG_COLORS['en'], LANG_COLORS['other']],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Distribución Lingüística del Corpus\n(sesgo: modelo entrenado principalmente en español)',
                  fontweight='bold')

# I4b: Distribución longitud por sentimiento (desigualdad estructural)
df_s['sentiment'] = pd.cut(df_s['score'], bins=[0,6,8,10], labels=['Negativo','Neutro','Positivo'])
bp_data = [df_s[df_s['sentiment']==s]['n_words'].clip(upper=60).dropna().values
           for s in ['Negativo','Neutro','Positivo']]
bp = axes[1].boxplot(bp_data, labels=['Negativo','Neutro','Positivo'],
                     patch_artist=True, showmeans=True)
for patch, color in zip(bp['boxes'], ['#F44336','#FFC107','#4CAF50']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
means = [df_s[df_s['sentiment']==s]['n_words'].mean() for s in ['Negativo','Neutro','Positivo']]
for i, m in enumerate(means):
    axes[1].text(i+1, m+0.5, f'μ={m:.1f}', ha='center', fontsize=8, color='red')
axes[1].set_title('Longitud del Comentario por Clase\n(sesgo de longitud: negativos son sistemáticamente más largos)',
                  fontweight='bold')
axes[1].set_ylabel('N.º palabras (cap. 60)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'I4_sesgo_linguistico_longitud.png'), dpi=150)
plt.close()
print('Fig I4 guardada ✓')

Fig I4 guardada ✓


In [7]:
from matplotlib.patches import FancyBboxPatch
import matplotlib.pyplot as plt
import os

# ══════════════════════════════════════════════════════════════════════════
# FIG I5 — RESUMEN VISUAL DE SESGOS
# ══════════════════════════════════════════════════════════════════════════

sesgos = [
    (
        '1. Truncamiento\nde plataforma',
        'Booking omite scores 2 y 4',
        'Alto',
        'Evaluar modelo por plataforma\nno usar fuente como feature'
    ),
    (
        '2. Positividad\n(Survivorship)',
        '59% de reseñas score 9-10\nno es muestra aleatoria de huéspedes',
        'Alto',
        'class_weight=balanced\nSMOTE en clase Negativo'
    ),
    (
        '3. Geográfico',
        'Solo 3 ciudades españolas\n63 hoteles de Madrid sin reseñas',
        'Medio',
        'Estratificar split por ciudad\nNo generalizar a otras ciudades'
    ),
    (
        '4. Temporal',
        'Solo Hotels.com tiene fecha\nBooking y Expedia: 0% cobertura',
        'Medio',
        'No usar fecha como feature\nAnálisis temporal solo descriptivo'
    ),
    (
        '5. Lingüístico',
        '~68% español, ~4% inglés\n~28% idiomas mixtos/otros',
        'Bajo',
        'Stopwords mixtas (es+en)\nMonitorizar rendimiento en inglés'
    ),
    (
        '6. Longitud\nde texto',
        'Negativos: μ=22 palabras\nPositivos: μ=16 palabras',
        'Bajo',
        'Incluir n_palabras como feature\nNo eliminar comentarios largos'
    ),
    (
        '7. Muestreo\nuniforme',
        'Media 6.4 reseñas/hotel\nmáx. 15 → no refleja popularidad real',
        'Bajo',
        'No usar hotel como feature de identidad\n(riesgo de overfitting por hotel)'
    ),
]

severity_color = {
    'Alto': '#F44336',
    'Medio': '#FFC107',
    'Bajo': '#4CAF50'
}

fig, ax = plt.subplots(figsize=(16, 9))
fig.suptitle(
    'I5 — Mapa de Sesgos: Resumen Visual',
    fontsize=16,
    fontweight='bold'
)

ax.axis('off')

headers = ['Sesgo', 'Descripción', 'Impacto', 'Mitigación']

col_x = [0.01, 0.20, 0.64, 0.75]
col_w = [0.18, 0.43, 0.10, 0.24]

# ─────────────────────────────────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────────────────────────────────

for hx, hw, ht in zip(col_x, col_w, headers):

    ax.add_patch(
        plt.Rectangle(
            (hx, 0.92),
            hw,
            0.06,
            transform=ax.transAxes,
            color='#1F4E79',
            zorder=2
        )
    )

    ax.text(
        hx + hw / 2,
        0.95,
        ht,
        transform=ax.transAxes,
        ha='center',
        va='center',
        fontsize=11,
        color='white',
        fontweight='bold'
    )

# ─────────────────────────────────────────────────────────────────────────
# ROWS
# ─────────────────────────────────────────────────────────────────────────

row_h = 0.12

for i, (name, desc, impact, action) in enumerate(sesgos):

    y = 0.90 - (i + 1) * row_h

    bg = '#F5F9FD' if i % 2 == 0 else '#FFFFFF'

    # Background row
    ax.add_patch(
        plt.Rectangle(
            (0.01, y),
            0.98,
            row_h - 0.01,
            transform=ax.transAxes,
            color=bg,
            zorder=1
        )
    )

    # Column 1 — Sesgo
    ax.text(
        col_x[0] + col_w[0] / 2,
        y + row_h / 2,
        name,
        transform=ax.transAxes,
        ha='center',
        va='center',
        fontsize=9,
        fontweight='bold'
    )

    # Column 2 — Descripción
    ax.text(
        col_x[1] + 0.01,
        y + row_h / 2,
        desc,
        transform=ax.transAxes,
        ha='left',
        va='center',
        fontsize=8
    )

    # Column 3 — Impacto badge
    badge_x = col_x[2] + col_w[2] / 2

    ax.add_patch(
        FancyBboxPatch(
            (badge_x - 0.04, y + row_h / 2 - 0.025),
            0.08,
            0.05,
            boxstyle='round,pad=0.005',
            transform=ax.transAxes,
            color=severity_color[impact],
            zorder=3
        )
    )

    ax.text(
        badge_x,
        y + row_h / 2,
        impact,
        transform=ax.transAxes,
        ha='center',
        va='center',
        fontsize=8,
        color='white',
        fontweight='bold'
    )

    # Column 4 — Mitigación
    ax.text(
        col_x[3] + 0.01,
        y + row_h / 2,
        action,
        transform=ax.transAxes,
        ha='left',
        va='center',
        fontsize=7.5,
        color='#1F4E79'
    )

# ─────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, 'I5_mapa_sesgos.png'),
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print('Fig I5 guardada ✓')

Fig I5 guardada ✓


---
## Bloque J — Preprocesamiento

El pipeline de preprocesamiento transforma el dataset bruto en un corpus limpio y vectorizable.
Cada paso está documentado con su impacto en el número de reseñas.

In [8]:
# ══════════════════════════════════════════════════════════════════════════
# PIPELINE DE PREPROCESAMIENTO — PASO A PASO
# ══════════════════════════════════════════════════════════════════════════

STOP_ES = set("""
a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante
e el ella ellas ello ellos en entre era es esa ese eso esta este esto fue han hasta he la
las le les lo los me mi más no nos o os para pero por que quien se si sin sobre son su sus
también te todo todos tu tus un una unas unos ya yo muy bien bueno buena hay ser estar
tener había hacer haber fueron fueron
""".split())

STOP_EN = set("""
the a an and or but in on at to for of with is was are were be been have has had do does did
will would could should may might this that these those it its i you he she we they me him
her us them my your his our their all any both each few more most other some such no nor only
own same than then very just as also so if about after before between because room hotel
""".split())

STOPWORDS = STOP_ES | STOP_EN

# Regex para limpiar prefijo numérico de Hotels.com
PREFIX_PAT = re.compile(r'^\d+/\d+\s+\w+\s+\d+\s+de\s+\d+\s+\w+\s+', re.IGNORECASE)

def tokenize(text):
    """Tokenización con limpieza completa."""
    text = PREFIX_PAT.sub('', str(text)).strip()   # quitar prefijo Hotels.com
    text = text.lower()                             # minúsculas
    text = re.sub(r'\d+/\d+', '', text)            # quitar fracciones numéricas
    text = re.sub(r'[^\w\sáéíóúüñà]', ' ', text)  # quitar puntuación
    text = re.sub(r'\s+', ' ', text).strip()        # normalizar espacios
    tokens = [w for w in text.split()
              if w not in STOPWORDS and len(w) > 2]
    return tokens

# ── Ejecutar pipeline ────────────────────────────────────────────────────
pipeline_log = []
df = df_raw.copy()
pipeline_log.append(('0 — Dataset bruto', len(df)))

# Paso 1: eliminar score NaN
df = df.dropna(subset=['score'])
pipeline_log.append(('1 — Drop score NaN', len(df)))

# Paso 2: eliminar comentarios < 5 caracteres
df = df[df['comment'].str.len() >= 5].copy()
pipeline_log.append(('2 — Drop comment < 5 chars', len(df)))

# Paso 3: deduplicación (mismo hotel + mismo comentario)
df = df.drop_duplicates(subset=['hotel', 'comment'], keep='first').copy()
pipeline_log.append(('3 — Deduplicación (hotel+comment)', len(df)))

# Paso 4: tokenización + limpieza
df['tokens'] = df['comment'].apply(tokenize)
df['n_tokens'] = df['tokens'].str.len()
df['text_clean'] = df['tokens'].str.join(' ')

# Paso 5: eliminar reseñas que quedan vacías tras tokenización
df = df[df['n_tokens'] > 0].copy()
pipeline_log.append(('4+5 — Tokenización + drop vacíos', len(df)))

# Paso 6: variable objetivo
df['sentiment'] = pd.cut(df['score'], bins=[0, 6, 8, 10],
                         labels=['Negativo', 'Neutro', 'Positivo'])
pipeline_log.append(('6 — Variable objetivo asignada', len(df)))

# Paso 7: features estructurales
df['n_words_log'] = np.log1p(df['n_tokens'])  # longitud log-normalizada
df['price_norm'] = df.groupby('city')['price'].transform(
    lambda x: (x - x.median()) / (x.quantile(0.75) - x.quantile(0.25) + 1e-9)
)  # precio normalizado por ciudad (IQR)

pipeline_log.append(('7 — Features estructurales', len(df)))

print('=== LOG DEL PIPELINE ===')
for step, n in pipeline_log:
    print(f'  {step:<40}: {n:>5} reseñas')

print(f'\n📊 Dataset final: {len(df)} reseñas')
print(f'   Eliminadas: {len(df_raw) - len(df)} ({(len(df_raw)-len(df))/len(df_raw)*100:.1f}%)')
print(f'\nDistribución target:')
for s in ['Negativo','Neutro','Positivo']:
    n = (df['sentiment']==s).sum()
    print(f'  {s}: {n} ({n/len(df)*100:.1f}%)')

=== LOG DEL PIPELINE ===
  0 — Dataset bruto                       :  3881 reseñas
  1 — Drop score NaN                      :  3877 reseñas
  2 — Drop comment < 5 chars              :  3876 reseñas
  3 — Deduplicación (hotel+comment)       :  3860 reseñas
  4+5 — Tokenización + drop vacíos        :  3777 reseñas
  6 — Variable objetivo asignada          :  3777 reseñas
  7 — Features estructurales              :  3777 reseñas

📊 Dataset final: 3777 reseñas
   Eliminadas: 104 (2.7%)

Distribución target:
  Negativo: 530 (14.0%)
  Neutro: 1028 (27.2%)
  Positivo: 2219 (58.8%)


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# FIG J1 — IMPACTO DEL PIPELINE SOBRE EL TAMAÑO DEL CORPUS
# ══════════════════════════════════════════════════════════════════════════
steps  = [s for s, _ in pipeline_log]
counts = [n for _, n in pipeline_log]
losses = [counts[0] - counts[i] for i in range(len(counts))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('J1 — Impacto del Pipeline de Preprocesamiento sobre el Corpus',
             fontsize=14, fontweight='bold')

# J1a: Evolución del tamaño en cada paso
colors_pipe = ['#5C6BC0' if i == 0 else '#4CAF50' if i == len(counts)-1 else '#78909C'
               for i in range(len(counts))]
bars = axes[0].barh(range(len(steps)), counts, color=colors_pipe, edgecolor='white')
axes[0].set_yticks(range(len(steps)))
axes[0].set_yticklabels(steps, fontsize=8)
axes[0].set_xlabel('N.º reseñas')
axes[0].set_title('Reseñas en Cada Paso del Pipeline', fontweight='bold')
axes[0].set_xlim(3500, 3950)
for bar, val in zip(bars, counts):
    axes[0].text(val + 3, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=8)

# J1b: Distribución target final
sent_counts = df['sentiment'].value_counts().reindex(['Negativo','Neutro','Positivo'])
sent_pct = sent_counts / len(df) * 100
bars2 = axes[1].bar(['Negativo','Neutro','Positivo'], sent_pct.values,
                    color=['#F44336','#FFC107','#4CAF50'], edgecolor='white', width=0.5)
for bar, (val, pct) in zip(bars2, zip(sent_counts.values, sent_pct.values)):
    axes[1].text(bar.get_x() + bar.get_width()/2, pct + 0.5,
                 f'{val}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylim(0, 80)
axes[1].set_title('Distribución Target Final\n(dataset desequilibrado → requiere balanceo)',
                  fontweight='bold')
axes[1].set_ylabel('% del corpus')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'J1_impacto_pipeline.png'), dpi=150)
plt.close()
print('Fig J1 guardada ✓')

Fig J1 guardada ✓


In [10]:
# ══════════════════════════════════════════════════════════════════════════
# FIG J2 — TOKENIZACIÓN: ANÁLISIS DEL VOCABULARIO
# ══════════════════════════════════════════════════════════════════════════
from collections import Counter

all_tokens = [tok for toks in df['tokens'] for tok in toks]
vocab = Counter(all_tokens)
vocab_size = len(vocab)
freq_dist = sorted(vocab.values(), reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('J2 — Análisis del Vocabulario Post-Tokenización', fontsize=16, fontweight='bold')

# J2a: Top 20 tokens
top20 = vocab.most_common(20)
t_words, t_counts = zip(*top20)
axes[0].barh(t_words[::-1], t_counts[::-1], color='#3F51B5', edgecolor='white')
axes[0].set_title(f'Top 20 Tokens\n(vocabulario total: {vocab_size:,} términos únicos)',
                  fontweight='bold')
axes[0].set_xlabel('Frecuencia')

# J2b: Distribución de frecuencias (curva Zipf)
axes[1].loglog(range(1, len(freq_dist)+1), freq_dist, color='#E91E63', linewidth=1.5)
axes[1].set_title('Distribución de Frecuencias (escala log-log)\n→ sigue ley de Zipf, normal en corpus NLP',
                  fontweight='bold')
axes[1].set_xlabel('Rango del término')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(alpha=0.3)

# J2c: Histograma n_tokens por reseña
axes[2].hist(df['n_tokens'].clip(upper=50), bins=25, color='#009688', edgecolor='white')
axes[2].axvline(df['n_tokens'].mean(), color='red', linestyle='--',
                label=f'Media: {df["n_tokens"].mean():.1f} tokens')
axes[2].axvline(df['n_tokens'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df["n_tokens"].median():.0f} tokens')
axes[2].set_title('N.º de Tokens por Reseña (cap. 50)\ntras eliminar stopwords', fontweight='bold')
axes[2].set_xlabel('N.º tokens')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'J2_vocabulario.png'), dpi=150)
plt.close()
print(f'Fig J2 guardada ✓')
print(f'\n📊 Vocabulario:')
print(f'   Términos únicos: {vocab_size:,}')
print(f'   Total tokens:    {len(all_tokens):,}')
print(f'   Media tokens/reseña: {df["n_tokens"].mean():.1f}')
print(f'   Mediana: {df["n_tokens"].median():.0f}')

Fig J2 guardada ✓

📊 Vocabulario:
   Términos únicos: 3,881
   Total tokens:    30,325
   Media tokens/reseña: 8.0
   Mediana: 5


In [11]:
# ══════════════════════════════════════════════════════════════════════════
# FIG J3 — FEATURES ESTRUCTURALES: DISTRIBUCIÓN Y NORMALIZACIÓN
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('J3 — Features Estructurales: Distribución y Normalización',
             fontsize=16, fontweight='bold')

# J3a: n_words_log (longitud log-normalizada)
for sent, color in SENT_COLORS.items():
    sub = df[df['sentiment']==sent]['n_words_log']
    if len(sub) > 0:
        axes[0].hist(sub, bins=20, alpha=0.5, color=color, label=sent, density=True)
axes[0].set_title('Longitud (log) por Clase de Sentimiento\n→ separación visible entre Negativo y Positivo',
                  fontweight='bold')
axes[0].set_xlabel('log(1 + n_tokens)')
axes[0].legend()

# J3b: precio normalizado por ciudad
for city, color in CITY_COLORS.items():
    sub = df[df['city']==city]['price_norm'].dropna().clip(-3, 4)
    axes[1].hist(sub, bins=20, alpha=0.5, color=color, label=city, density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Precio Normalizado por Ciudad (IQR)\n→ comparable entre Barcelona, Madrid, Bilbao',
                  fontweight='bold')
axes[1].set_xlabel('(precio − mediana) / IQR')
axes[1].legend()

# J3c: impacto imputación precio
price_na = df['price'].isna().sum()
price_ok = df['price'].notna().sum()
axes[2].pie(
    [price_ok, price_na],
    labels=[f'Con precio\n({price_ok}, {price_ok/len(df)*100:.1f}%)',
            f'Imputado con mediana ciudad\n({price_na}, {price_na/len(df)*100:.1f}%)'],
    colors=['#4CAF50','#FFC107'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
axes[2].set_title('Completitud del Campo Precio\n(valores ausentes imputados con mediana por ciudad)',
                  fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'J3_features_estructurales.png'), dpi=150)
plt.close()
print('Fig J3 guardada ✓')

Fig J3 guardada ✓


In [12]:
# ══════════════════════════════════════════════════════════════════════════
# GUARDAR DATASET PROCESADO
# ══════════════════════════════════════════════════════════════════════════
output_cols = ['hotel','city','source','comment','text_clean','tokens',
               'score','sentiment','n_tokens','n_words_log','price','price_norm',
               'n_services','date']
df_out = df[[c for c in output_cols if c in df.columns]].copy()

# Guardar JSON y CSV
df_out.drop(columns=['tokens']).to_csv(
    os.path.join(OUTPUT_DIR, 'corpus_procesado.csv'), index=False, encoding='utf-8')

print('=== DATASET PROCESADO GUARDADO ===')
print(f'Archivo: corpus_procesado.csv')
print(f'Filas: {len(df_out)}')
print(f'Columnas: {list(df_out.columns)}')

print(f'\n✅ Todas las figuras guardadas en: {OUTPUT_DIR}')
for f in ['I1_sesgo_plataforma_scores','I2_sesgo_positividad',
          'I3_sesgo_geografico_cobertura','I4_sesgo_linguistico_longitud',
          'I5_mapa_sesgos','J1_impacto_pipeline','J2_vocabulario',
          'J3_features_estructurales']:
    print(f'   {f}.png')

=== DATASET PROCESADO GUARDADO ===
Archivo: corpus_procesado.csv
Filas: 3777
Columnas: ['hotel', 'city', 'source', 'comment', 'text_clean', 'tokens', 'score', 'sentiment', 'n_tokens', 'n_words_log', 'price', 'price_norm', 'n_services', 'date']

✅ Todas las figuras guardadas en: /Users/alessiadelconte/codici/progetto NPL/output_sesgos_preprocesamiento
   I1_sesgo_plataforma_scores.png
   I2_sesgo_positividad.png
   I3_sesgo_geografico_cobertura.png
   I4_sesgo_linguistico_longitud.png
   I5_mapa_sesgos.png
   J1_impacto_pipeline.png
   J2_vocabulario.png
   J3_features_estructurales.png
